# Source-hour manifest (`07_source_hour_manifest`)

GH Archive delivers one file per hour. Seven days of timestamps do not prove that every hour was loaded. G5 (`release_activity_shift`) compares the 24 hours before a release with the 24 hours after it, so one missing hour would make it compare 23 hours with 24 without anyone noticing.

This notebook builds the proof: `github_lakehouse.audit.source_hour_manifest`, with one row per expected UTC hour. G5 reads only the column `is_complete`.

Contract: `docs/gold_contracts.md`, section 8.

| | |
|---|---|
| Reads | `bronze.github_events_raw`. With `check_s3 = yes` also the raw files in S3 |
| Writes | `audit.source_hour_manifest` (replaced completely by every successful run), its staging copy, and one row in `audit.gold_pipeline_runs` |
| Grain | One row per expected UTC hour |
| Key | `source_hour_utc` |

## When is an hour complete?

1. Its file is in Bronze with at least one row, **and**
2. if the file was compared with the raw file in S3, both have the same number of rows.

An hour with zero rows is not complete. A real GH Archive hour has many thousands of events.

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | Create the table | `CREATE TABLE IF NOT EXISTS` with the columns of the contract |
| 3 | Expected hours | 24 hours for every day from the first to the last source date in Bronze |
| 4 | Bronze evidence | Rows and first ingestion time of every file in Bronze |
| 5 | S3 evidence | The number of lines of every raw file. Counted only on request, then kept |
| 6 | Put it together | One row per expected hour with all evidence and the `is_complete` flag |
| 7 | Stage | Writes the result to the staging copy |
| 8 | Quality gates | Checks the staging copy |
| 9 | Publish | Replaces the manifest with the staged rows, only if the gates passed |
| 10 | Quality numbers | Counts for the audit row, including the global routing check |
| 11 | Audit | Writes one audit row. Stops with an error if the run failed |
| 12 | Results | Shows the manifest |

Before the first run, `08_gold_audit_table` must have run once.

## 1. Parameters

The notebook has the three widgets every Gold notebook has, and one extra widget.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | The routing check in step 10 reads all of Bronze | `YYYY-MM-DD`: the routing check reads only that source date. The manifest itself is always rebuilt for all hours |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>` |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |
| `check_s3` | The raw files are not read | `yes`: the lines of every raw file in S3 are counted and compared with Bronze |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_source_hour_manifest"

# The folder with the raw GH Archive files. It is read only when check_s3 = yes.
RAW_BASE_PATH = "s3://tahmid-gharchive-lakehouse/raw/github/"

# An hour is shown as "thin" when it has less than this percentage
# of the average of the hour before and the hour after.
THIN_HOUR_LIMIT_PCT = 70

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
    processing_scope = "ALL_DATES"
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

    processing_scope = "SINGLE_DATE"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The four table names are set **only** in this cell. Later cells never change them, otherwise a test run could write to production.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    BRONZE_TABLE = "github_lakehouse.bronze.github_events_raw"
    MANIFEST_TABLE = "github_lakehouse.audit.source_hour_manifest"
    STAGING_TABLE = "github_lakehouse.gold_staging.source_hour_manifest"
    S3_COUNTS_TABLE = "github_lakehouse.gold_staging.source_hour_s3_line_counts"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    BRONZE_TABLE = f"github_lakehouse.{test_schema}.github_events_raw"
    MANIFEST_TABLE = f"github_lakehouse.{test_schema}.source_hour_manifest"
    STAGING_TABLE = f"github_lakehouse.{test_schema}.source_hour_manifest_staging"
    S3_COUNTS_TABLE = f"github_lakehouse.{test_schema}.source_hour_s3_line_counts"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

print("Bronze table  :", BRONZE_TABLE)
print("Manifest table:", MANIFEST_TABLE)
print("Staging table :", STAGING_TABLE)
print("Audit table   :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. The ID is written into every manifest row, so it may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 `check_s3`

`yes` counts the lines of every raw file in S3. This reads all raw files once and takes several minutes, so it is done only on request. A test run has no raw files, so `check_s3 = yes` is not allowed together with `test_schema`.

In [0]:
dbutils.widgets.text("check_s3", "")
check_s3 = dbutils.widgets.get("check_s3").strip().lower()

if check_s3 != "" and check_s3 != "yes":
    raise ValueError(
        f"Invalid check_s3 '{check_s3}'. Use yes, or leave it blank."
    )

if check_s3 == "yes" and test_schema != "":
    raise ValueError(
        "check_s3 = yes reads the real raw files in S3. "
        "It cannot be used together with test_schema."
    )

print("check_s3:", check_s3)

### 1.6 Is the setup done?

The audit table is created by `08_gold_audit_table`. If it does not exist, the notebook stops here with a clear message.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

print("Setup found:", AUDIT_TABLE)

## 2. Create the manifest table

The columns and types are those of the contract. `CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, so this step is safe to run again.

| Column | Meaning |
|---|---|
| `source_hour_utc` | Start of the hour. Key |
| `source_date` | The UTC date of the hour |
| `expected_file_name` | For example `2024-02-01-0.json.gz` |
| `bronze_ingested` | The file has rows in Bronze |
| `bronze_row_count` | Rows of that file in Bronze. 0 when the file is not in Bronze |
| `bronze_first_ingested_at` | First ingestion time of that file |
| `s3_file_found` | The raw file was found in S3. NULL when S3 was never compared for this hour |
| `s3_line_count` | Lines of the raw file |
| `counts_match` | Bronze rows equal raw lines. NULL when there is no raw line count to compare with |
| `evidence_level` | `BRONZE_AND_S3` when the raw file was found and counted, otherwise `BRONZE_ONLY` |
| `is_complete` | The one column G5 reads |
| `checked_at` | When the row was written |
| `run_id` | The run that wrote it |

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {MANIFEST_TABLE} (
        source_hour_utc           TIMESTAMP,
        source_date               DATE,
        expected_file_name        STRING,
        bronze_ingested           BOOLEAN,
        bronze_row_count          BIGINT,
        bronze_first_ingested_at  TIMESTAMP,
        s3_file_found             BOOLEAN,
        s3_line_count             BIGINT,
        counts_match              BOOLEAN,
        evidence_level            STRING,
        is_complete               BOOLEAN,
        checked_at                TIMESTAMP,
        run_id                    STRING
    )
    USING DELTA
""")

print("Table ready:", MANIFEST_TABLE)

## 3. Expected hours

### 3.1 The first and the last source date in Bronze

The expected hours are 24 for every day from the first to the last source date. A day in between that is missing completely is therefore still expected, and its 24 hours will show up as not complete.

`datediff(last, first) + 1` is the number of days. For 1 to 7 February it is 7, so 168 hours are expected.

In [0]:
bronze_scope = spark.sql(f"""
    SELECT
        COUNT(*)                                            AS bronze_rows,
        MIN(_source_date)                                   AS first_source_date,
        MAX(_source_date)                                   AS last_source_date,
        datediff(MAX(_source_date), MIN(_source_date)) + 1  AS expected_days
    FROM {BRONZE_TABLE}
""")
bronze_scope.createOrReplaceTempView("bronze_scope")

scope_row = bronze_scope.collect()[0]
bronze_row_total = scope_row["bronze_rows"]

# An empty Bronze table has no dates, so no hour is expected.
if scope_row["expected_days"] is None:
    expected_hour_count = 0
else:
    expected_hour_count = 24 * scope_row["expected_days"]

print("Bronze rows      :", bronze_row_total)
print("First source date:", scope_row["first_source_date"])
print("Last source date :", scope_row["last_source_date"])
print("Expected hours   :", expected_hour_count)

### 3.2 One row per hour

`sequence` makes a list of hours, from midnight of the first day to 23:00 of the last day. `explode` turns that list into rows, one row per hour.

When Bronze is empty, there is no first or last date and this step gives no rows.

In [0]:
hour_list = spark.sql("""
    SELECT explode(sequence(
               CAST(first_source_date AS TIMESTAMP),
               CAST(last_source_date AS TIMESTAMP) + INTERVAL 23 HOURS,
               INTERVAL 1 HOUR
           )) AS source_hour_utc
    FROM bronze_scope
""")
hour_list.createOrReplaceTempView("hour_list")

### 3.3 The file name of each hour

GH Archive names its files `YYYY-MM-DD-H.json.gz`. The hour has no leading zero: the file for 05:00 on 1 February is `2024-02-01-5.json.gz`.

In [0]:
expected_hours = spark.sql("""
    SELECT
        source_hour_utc,
        CAST(source_hour_utc AS DATE) AS source_date,
        concat(
            date_format(source_hour_utc, 'yyyy-MM-dd'),
            '-',
            CAST(hour(source_hour_utc) AS STRING),
            '.json.gz'
        ) AS expected_file_name
    FROM hour_list
""")
expected_hours.createOrReplaceTempView("expected_hours")

## 4. Bronze evidence

One row per file in Bronze: how many rows it has and when it was first ingested.

In [0]:
bronze_files = spark.sql(f"""
    SELECT
        _source_file_name  AS file_name,
        COUNT(*)           AS bronze_row_count,
        MIN(_ingested_at)  AS bronze_first_ingested_at
    FROM {BRONZE_TABLE}
    GROUP BY _source_file_name
""")
bronze_files.createOrReplaceTempView("bronze_files")

## 5. S3 evidence

### 5.1 Count the lines of the raw files (only with `check_s3 = yes`)

Every line of a raw file is one event. The files are read in the same way as in the Bronze ingestion notebook, and the lines are counted per file. The result is stored in a small staging table.

Reading from S3 can fail, for example when the bucket cannot be reached. In that case the notebook prints a warning and continues with the evidence it already has. It does not fail.

In [0]:
s3_check_done = False
s3_check_error = None

if check_s3 == "yes":
    try:
        raw_lines = (
            spark.read
            .option("recursiveFileLookup", "true")
            .text(RAW_BASE_PATH)
            .selectExpr("_metadata.file_name AS file_name")
        )
        raw_lines.createOrReplaceTempView("raw_lines")

        spark.sql(f"""
            CREATE OR REPLACE TABLE {S3_COUNTS_TABLE} AS
            SELECT
                file_name,
                COUNT(*) AS s3_line_count
            FROM raw_lines
            GROUP BY file_name
        """)

        s3_check_done = True
        print("Raw files counted:", S3_COUNTS_TABLE)

    except Exception as error:
        s3_check_error = str(error)[:500]
        print("WARNING: the raw files in S3 could not be read.")
        print("The manifest keeps the S3 evidence it already has.")
        print(s3_check_error)
else:
    print("check_s3 is blank: the raw files are not read in this run.")

### 5.2 The S3 evidence of this run

There are two cases.

- **The raw files were counted in this run.** Every expected hour gets fresh evidence: was its file found, and how many lines does it have.
- **They were not counted in this run.** The evidence of the last comparison is taken from the manifest as it is now. That is why a comparison has to run only once. Hours that were never compared have no evidence.

In both cases `counts_match` is calculated again in step 6 with the Bronze rows of today.

In [0]:
if s3_check_done:
    s3_evidence = spark.sql(f"""
        SELECT
            h.expected_file_name      AS file_name,
            c.file_name IS NOT NULL   AS s3_file_found,
            c.s3_line_count
        FROM expected_hours h
        LEFT JOIN {S3_COUNTS_TABLE} c
          ON c.file_name = h.expected_file_name
    """)
else:
    s3_evidence = spark.sql(f"""
        SELECT
            expected_file_name  AS file_name,
            s3_file_found,
            s3_line_count
        FROM {MANIFEST_TABLE}
        WHERE s3_file_found IS NOT NULL
    """)

s3_evidence.createOrReplaceTempView("s3_evidence")

## 6. Put it together

### 6.1 Join the evidence to the expected hours

The expected hours are on the left side of both joins, so every expected hour stays in the result, also when it has no file in Bronze. Such an hour gets 0 Bronze rows.

In [0]:
hour_evidence = spark.sql("""
    SELECT
        h.source_hour_utc,
        h.source_date,
        h.expected_file_name,
        COALESCE(b.bronze_row_count, 0)  AS bronze_row_count,
        b.bronze_first_ingested_at,
        s.s3_file_found,
        s.s3_line_count
    FROM expected_hours h
    LEFT JOIN bronze_files b
      ON b.file_name = h.expected_file_name
    LEFT JOIN s3_evidence s
      ON s.file_name = h.expected_file_name
""")
hour_evidence.createOrReplaceTempView("hour_evidence")

### 6.2 Compare Bronze with S3

- `bronze_ingested`: the file has at least one row in Bronze.
- `counts_match`: only when the raw file was found. Otherwise it stays NULL, because there is nothing to compare with.
- `evidence_level`: `BRONZE_AND_S3` when the raw file was found and counted, otherwise `BRONZE_ONLY`.

In [0]:
hour_checks = spark.sql("""
    SELECT
        source_hour_utc,
        source_date,
        expected_file_name,
        bronze_row_count,
        bronze_first_ingested_at,
        s3_file_found,
        s3_line_count,
        bronze_row_count > 0 AS bronze_ingested,
        CASE
            WHEN s3_file_found = true THEN bronze_row_count = s3_line_count
        END AS counts_match,
        CASE
            WHEN s3_file_found = true THEN 'BRONZE_AND_S3'
            ELSE 'BRONZE_ONLY'
        END AS evidence_level
    FROM hour_evidence
""")
hour_checks.createOrReplaceTempView("hour_checks")

### 6.3 The final rows

The columns are in the order of the contract.

`is_complete` is true when the file is in Bronze and the counts do not contradict it. `COALESCE(counts_match, true)` means: when there was nothing to compare with, the Bronze evidence alone decides.

In [0]:
manifest_rows = spark.sql(f"""
    SELECT
        source_hour_utc,
        source_date,
        expected_file_name,
        bronze_ingested,
        CAST(bronze_row_count AS BIGINT)                  AS bronze_row_count,
        bronze_first_ingested_at,
        s3_file_found,
        CAST(s3_line_count AS BIGINT)                     AS s3_line_count,
        counts_match,
        evidence_level,
        bronze_ingested AND COALESCE(counts_match, true)  AS is_complete,
        current_timestamp()                               AS checked_at,
        '{run_id}'                                        AS run_id
    FROM hour_checks
""")
manifest_rows.createOrReplaceTempView("manifest_rows")

## 7. Stage

The result is written to the staging copy first. The manifest itself is not touched yet.

Spark does its work only now, when the rows are written. If something goes wrong, the error is kept in `stage_error`, so that the audit row at the end can report it.

In [0]:
stage_error = None

try:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {STAGING_TABLE} AS
        SELECT * FROM manifest_rows
    """)
    print("Staged:", STAGING_TABLE)

except Exception as error:
    stage_error = str(error)[:1000]
    print("Staging FAILED:", stage_error)

## 8. Quality gates

The checks run on the staging copy. One query counts everything the checks need.

| Gate | It fails when |
|---|---|
| `ROW_COUNT_IS_NOT_24_PER_DAY` | The number of rows is not 24 times the number of expected days |
| `KEY_NOT_UNIQUE` | An hour appears more than once |
| `FILE_NAME_NOT_UNIQUE` | A file name appears more than once |
| `KEY_IS_NULL` | A row has no hour or no file name |
| `FLAG_IS_NULL` | `bronze_ingested`, `bronze_row_count`, `evidence_level` or `is_complete` is missing |
| `COMPLETE_WITHOUT_ROWS` | An hour is marked complete but has 0 rows in Bronze |
| `COMPLETE_WITH_COUNT_MISMATCH` | An hour is marked complete although Bronze and S3 have different counts |

Hours that are not complete do **not** fail a gate. They are a result, and G5 deals with them.

In [0]:
failed_checks = []
staged_row_count = 0

if stage_error is None:
    gate_row = spark.sql(f"""
        SELECT
            COUNT(*)                            AS staged_rows,
            COUNT(DISTINCT source_hour_utc)     AS different_hours,
            COUNT(DISTINCT expected_file_name)  AS different_file_names,
            count_if(source_hour_utc IS NULL OR expected_file_name IS NULL)
                                                AS rows_without_key,
            count_if(bronze_ingested IS NULL OR bronze_row_count IS NULL
                     OR evidence_level IS NULL OR is_complete IS NULL)
                                                AS rows_without_flag,
            count_if(is_complete = true AND bronze_row_count = 0)
                                                AS complete_rows_without_data,
            count_if(is_complete = true AND counts_match = false)
                                                AS complete_rows_with_mismatch
        FROM {STAGING_TABLE}
    """).collect()[0]

    staged_row_count = gate_row["staged_rows"]

    # With no staged rows there is nothing to check. The run ends as NO_INPUT.
    if staged_row_count > 0:
        if staged_row_count != expected_hour_count:
            failed_checks.append("ROW_COUNT_IS_NOT_24_PER_DAY")
        if gate_row["different_hours"] != staged_row_count:
            failed_checks.append("KEY_NOT_UNIQUE")
        if gate_row["different_file_names"] != staged_row_count:
            failed_checks.append("FILE_NAME_NOT_UNIQUE")
        if gate_row["rows_without_key"] > 0:
            failed_checks.append("KEY_IS_NULL")
        if gate_row["rows_without_flag"] > 0:
            failed_checks.append("FLAG_IS_NULL")
        if gate_row["complete_rows_without_data"] > 0:
            failed_checks.append("COMPLETE_WITHOUT_ROWS")
        if gate_row["complete_rows_with_mismatch"] > 0:
            failed_checks.append("COMPLETE_WITH_COUNT_MISMATCH")

print("Staged rows  :", staged_row_count)
print("Expected rows:", expected_hour_count)

if len(failed_checks) == 0:
    print("Quality gates: no gate failed")
else:
    print("Quality gates FAILED:", "; ".join(failed_checks))

## 9. Publish

What happens with the staged rows:

| Situation | Status | Manifest |
|---|---|---|
| Staging failed | `FAILED` | Unchanged |
| No rows were staged, because Bronze is empty | `NO_INPUT` | Unchanged |
| A quality gate failed | `FAILED` | Unchanged |
| Everything is fine | `SUCCEEDED` | Replaced by the staged rows |

`INSERT OVERWRITE` replaces all rows of the manifest in one Delta commit. A reader sees either the old manifest or the new one, never a half-written one.

In [0]:
run_status = None
error_message = None
published_rows = 0

if stage_error is not None:
    run_status = "FAILED"
    error_message = "Staging failed: " + stage_error

elif staged_row_count == 0:
    run_status = "NO_INPUT"

elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Quality gates failed: " + "; ".join(failed_checks)

else:
    try:
        spark.sql(f"""
            INSERT OVERWRITE TABLE {MANIFEST_TABLE}
            SELECT
                source_hour_utc,
                source_date,
                expected_file_name,
                bronze_ingested,
                bronze_row_count,
                bronze_first_ingested_at,
                s3_file_found,
                s3_line_count,
                counts_match,
                evidence_level,
                is_complete,
                checked_at,
                run_id
            FROM {STAGING_TABLE}
        """)
        run_status = "SUCCEEDED"
        published_rows = staged_row_count

    except Exception as error:
        run_status = "FAILED"
        error_message = "Publish failed: " + str(error)[:1000]

print("Status:", run_status)
print("Published rows:", published_rows)

## 10. Quality numbers for the audit row

These numbers describe the run. None of them stops it.

### 10.1 Hours by result

The numbers are counted on the staging copy, so they describe what this run calculated, also when it was not published.

In [0]:
complete_hours = 0
incomplete_hours = 0
hours_in_bronze = 0
hours_compared_with_s3 = 0
hours_with_count_mismatch = 0

if stage_error is None:
    summary_row = spark.sql(f"""
        SELECT
            count_if(is_complete)           AS complete_hours,
            count_if(NOT is_complete)       AS incomplete_hours,
            count_if(bronze_ingested)       AS hours_in_bronze,
            count_if(s3_file_found)         AS hours_compared_with_s3,
            count_if(counts_match = false)  AS hours_with_count_mismatch
        FROM {STAGING_TABLE}
    """).collect()[0]

    complete_hours = summary_row["complete_hours"]
    incomplete_hours = summary_row["incomplete_hours"]
    hours_in_bronze = summary_row["hours_in_bronze"]
    hours_compared_with_s3 = summary_row["hours_compared_with_s3"]
    hours_with_count_mismatch = summary_row["hours_with_count_mismatch"]

print("Complete hours             :", complete_hours)
print("Hours that are not complete:", incomplete_hours)
print("Hours with a file in Bronze:", hours_in_bronze)
print("Hours compared with S3     :", hours_compared_with_s3)
print("Hours with different counts:", hours_with_count_mismatch)

### 10.2 The days in the manifest

The list of days goes into the column `affected_dates` of the audit row.

In [0]:
affected_dates = []

if stage_error is None:
    date_rows = spark.sql(f"""
        SELECT DISTINCT source_date
        FROM {STAGING_TABLE}
        ORDER BY source_date
    """).collect()

    for date_row in date_rows:
        affected_dates.append(date_row["source_date"])

print("Days in the manifest:", len(affected_dates))

### 10.3 Files in Bronze that no hour expects

The manifest has one row per expected hour. A file with another name would not appear in it, so it is counted here. Expected: 0.

In [0]:
unexpected_files = spark.sql("""
    SELECT
        b.file_name,
        b.bronze_row_count
    FROM bronze_files b
    LEFT JOIN expected_hours h
      ON h.expected_file_name = b.file_name
    WHERE h.expected_file_name IS NULL
""")
unexpected_files.createOrReplaceTempView("unexpected_files")

unexpected_file_count = unexpected_files.count()
print("Files in Bronze that no hour expects:", unexpected_file_count)

### 10.4 Thin hours

An hour can be in Bronze and still look wrong: it has far fewer rows than the hours around it. This step compares every hour with the hour before and the hour after. An hour is listed when it has less than 70% of the average of its two neighbours.

A thin hour stays complete. The file is loaded, it is just smaller than expected, and only a look inside the hour can tell why (step 12).

`NULLIF(..., 0)` protects the division: it turns a sum of 0 into NULL, and dividing by NULL gives NULL, not an error.

In [0]:
thin_hour_count = 0

if stage_error is None:
    thin_hours = spark.sql(f"""
        SELECT
            this_hour.source_hour_utc,
            this_hour.expected_file_name,
            this_hour.bronze_row_count,
            hour_before.bronze_row_count  AS rows_hour_before,
            hour_after.bronze_row_count   AS rows_hour_after,
            ROUND(
                200 * this_hour.bronze_row_count
                / NULLIF(hour_before.bronze_row_count + hour_after.bronze_row_count, 0),
                1
            ) AS pct_of_neighbours
        FROM {STAGING_TABLE} this_hour
        JOIN {STAGING_TABLE} hour_before
          ON hour_before.source_hour_utc = this_hour.source_hour_utc - INTERVAL 1 HOUR
        JOIN {STAGING_TABLE} hour_after
          ON hour_after.source_hour_utc = this_hour.source_hour_utc + INTERVAL 1 HOUR
        WHERE this_hour.bronze_row_count > 0
          AND hour_before.bronze_row_count > 0
          AND hour_after.bronze_row_count > 0
          AND 200 * this_hour.bronze_row_count
              < {THIN_HOUR_LIMIT_PCT} * (hour_before.bronze_row_count + hour_after.bronze_row_count)
    """)
    thin_hours.createOrReplaceTempView("thin_hours")

    thin_hour_count = thin_hours.count()

print("Thin hours:", thin_hour_count)

### 10.5 The global routing check

Every Silver notebook picks its rows with `get_json_object(raw_json, '$.type')`. A Bronze row whose JSON is broken, or which has no `type`, is picked by no Silver notebook and would disappear without a trace. This step counts such rows. The Silver profiling found 0.

With a `source_date` only that date is read. With a blank date all of Bronze is read.

In [0]:
if source_date_value is None:
    routing_filter = "true"
else:
    routing_filter = f"_source_date = DATE'{source_date}'"

routing_row = spark.sql(f"""
    SELECT
        COUNT(*)                                                 AS rows_checked,
        count_if(get_json_object(raw_json, '$.type') IS NULL)    AS rows_without_readable_type
    FROM {BRONZE_TABLE}
    WHERE {routing_filter}
""").collect()[0]

rows_checked_for_type = routing_row["rows_checked"]
rows_without_readable_type = routing_row["rows_without_readable_type"]

print("Bronze rows checked         :", rows_checked_for_type)
print("Rows without a readable type:", rows_without_readable_type)

if rows_without_readable_type > 0:
    print("WARNING: these rows are not read by any Silver notebook.")

## 11. Audit

One row is added to the Gold audit table for this run. The values are written in the order of the table's columns.

If the run failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

In [0]:
# The Delta version of the manifest after this run.
manifest_version = spark.sql(
    f"DESCRIBE HISTORY {MANIFEST_TABLE} LIMIT 1"
).collect()[0]["version"]

if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

# True when no gate failed. A run without rows has no failed gate.
checks_passed = stage_error is None and len(failed_checks) == 0

if check_s3 == "yes":
    s3_check_requested = 1
else:
    s3_check_requested = 0

if s3_check_done:
    s3_check_done_number = 1
else:
    s3_check_done_number = 0

quality_metrics = {
    "expected_hours": expected_hour_count,
    "complete_hours": complete_hours,
    "incomplete_hours": incomplete_hours,
    "hours_in_bronze": hours_in_bronze,
    "hours_compared_with_s3": hours_compared_with_s3,
    "hours_with_count_mismatch": hours_with_count_mismatch,
    "unexpected_bronze_files": unexpected_file_count,
    "thin_hours": thin_hour_count,
    "bronze_rows_checked_for_type": rows_checked_for_type,
    "bronze_rows_without_readable_type": rows_without_readable_type,
    "s3_check_requested": s3_check_requested,
    "s3_check_done": s3_check_done_number,
}

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                  # audit_id
    run_id,                             # run_id
    PIPELINE_NAME,                      # pipeline_name
    MANIFEST_TABLE,                     # gold_table
    processing_scope,                   # processing_scope
    source_date_value,                  # source_date
    affected_dates,                     # affected_dates
    run_started_at,                     # run_started_at
    run_finished_at,                    # run_finished_at
    run_status,                         # status
    error_message,                      # error_message
    {"bronze": bronze_row_total},       # source_rows_read
    {},                                 # rows_excluded
    published_rows,                     # gold_rows_published
    manifest_version,                   # gold_table_version
    checks_passed,                      # checks_passed
    failed_checks_text,                 # failed_checks
    quality_metrics,                    # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)
print("quality_metrics:", quality_metrics)

if run_status == "NO_INPUT":
    print("WARNING: Bronze has no rows. The manifest was not changed.")

if incomplete_hours > 0:
    print("WARNING:", incomplete_hours, "hours are not complete. See step 12.")

if run_status == "FAILED":
    raise RuntimeError("Source-hour manifest FAILED: " + error_message)

## 12. Results

These cells only read. They show the manifest as it is after this run.

### 12.1 Hours per day

Expected for 1 to 7 February 2024: 7 days with 24 hours each, all complete.

In [0]:
display(spark.sql(f"""
    SELECT
        source_date,
        COUNT(*)                        AS hours,
        count_if(is_complete)           AS complete_hours,
        count_if(NOT is_complete)       AS hours_not_complete,
        SUM(bronze_row_count)           AS bronze_rows,
        count_if(s3_file_found)         AS hours_compared_with_s3,
        count_if(counts_match = false)  AS hours_with_different_counts
    FROM {MANIFEST_TABLE}
    GROUP BY source_date
    ORDER BY source_date
"""))

### 12.2 Hours that are not complete

Expected: no rows.

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {MANIFEST_TABLE}
    WHERE is_complete = false
    ORDER BY source_hour_utc
"""))

### 12.3 Files in Bronze that no hour expects

Expected: no rows.

In [0]:
display(spark.sql("""
    SELECT *
    FROM unexpected_files
    ORDER BY file_name
"""))

### 12.4 Thin hours

Hours with less than 70% of the average of their two neighbours. The profiling run found one such hour: `2024-02-05-19.json.gz`.

In [0]:
display(spark.sql("""
    SELECT *
    FROM thin_hours
    ORDER BY pct_of_neighbours
"""))

### 12.5 A look inside the thinnest hour

For the thinnest hour of 12.4, this step counts the events of each minute. `created_at` looks like `2024-02-05T19:23:45Z`, so its first 16 characters are the minute.

How to read it:

- **Some minutes are missing or almost empty:** a part of the hour was not recorded by GH Archive.
- **All 60 minutes are there, each with fewer events than usual:** the whole hour was thin at the source.

If there is no thin hour, the cell prints a message and shows nothing.

In [0]:
thinnest_rows = spark.sql("""
    SELECT expected_file_name
    FROM thin_hours
    ORDER BY pct_of_neighbours
    LIMIT 1
""").collect()

if len(thinnest_rows) == 0:
    print("There is no thin hour.")
else:
    thinnest_file_name = thinnest_rows[0]["expected_file_name"]
    print("Events per minute in", thinnest_file_name)

    display(spark.sql(f"""
        SELECT
            substring(get_json_object(raw_json, '$.created_at'), 1, 16)  AS minute_utc,
            COUNT(*)                                                     AS events
        FROM {BRONZE_TABLE}
        WHERE _source_file_name = '{thinnest_file_name}'
        GROUP BY substring(get_json_object(raw_json, '$.created_at'), 1, 16)
        ORDER BY minute_utc
    """))

## What this notebook wrote

- `audit.source_hour_manifest`: replaced, when the status is `SUCCEEDED`.
- `gold_staging.source_hour_manifest`: the staging copy, overwritten on every run.
- `gold_staging.source_hour_s3_line_counts`: only with `check_s3 = yes`.
- `audit.gold_pipeline_runs`: one new row.

With a `test_schema`, all four are in that schema.